Sascha Spors,
Professorship Signal Theory and Digital Signal Processing,
Institute of Communications Engineering (INT),
Faculty of Computer Science and Electrical Engineering (IEF),
University of Rostock,
Germany

# Data Driven Audio Signal Processing - A Tutorial with Computational Examples

Master Course #24512

- lecture: https://github.com/spatialaudio/data-driven-audio-signal-processing-lecture
- tutorial: https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise

Feel free to contact lecturer frank.schultz@uni-rostock.de

# Homework Template for Matrix Fundamentals

In [ ]:
import numpy as np
import scipy as sp
from sympy import Matrix  # only needed for rref()
# we do the other matrix algebra with numpy arrays
# and scipy.linalg functions

# Example Matrix A

In [ ]:
A = np.array([
    [2, np.sqrt(5), 0],
    [0, 0, 1],
    [0, 0, 0],
    [0, 0, 0]
    ])
M, N = A.shape

In [ ]:
A, M, N

# Reduced Row Echelon Form

In [ ]:
R0 = Matrix(A).rref()[0]
R0  # A and R0 have rank 2

# SVD with Scipy

In [ ]:
U, s, Vh = sp.linalg.svd(A)
V = (Vh.T).conj()
S = sp.linalg.diagsvd(s, M, N)

In [ ]:
U

In [ ]:
S  # A has rank 2, because two non-zero singular values

In [ ]:
V

# Reduced SVD From Eigenvalue Problem

In [ ]:
De, Ve = sp.linalg.eig(A.T@A)
De = np.real(De)  # we know that eigen values are real and >=0
De, Ve  # A.T@A and thus also A has rank 2, because two non-zero eigen values

In [ ]:
V1 = Ve[:, 1][:, None]  # v vec for highest eigval
S1 = np.sqrt(De[1])  # highest singval
V2 = Ve[:, 2][:, None]  # v vec for lowest eigval
S2 = np.sqrt(De[2])  # lowest singval
U1 = A @ V1 / S1  # linked u vec
U2 = A @ V2 / S2  # linked u vec

Vred = np.hstack((V1, V2))  # rank 2
Ured = np.hstack((U1, U2))  # rank 2
Sred = np.array([[S1, 0],  # sorted from highest to lowest
                 [0, S2]])  # rank 2

A1 = Ured @ Sred @ Vred.T  # rank 2
np.allclose(A, A1)

In [ ]:
# check orthonormal vectors
Vred.T @ Vred, Ured.T @ Ured

In [ ]:
# match with scipy's SVD by polarity inversion
Ured[:, 0] *= -1
Vred[:, 0] *= -1
A1 = Ured @ Sred @ Vred.T  # rank 2
np.allclose(A, A1), np.allclose(Ured, U[:, :2]), np.allclose(Vred, V[:, :2])

# Full SVD From Eigenvalue Problem And Adding Null Spaces

With `Vred` and `Ured` already calculated above, we need to find the null space and the left null space and stack these vectors to full `V` and full `U`, respectively. For that we need orthonormal vectors, which the `null_space()` function delivers by default.

In [ ]:
Vf = np.hstack((Vred, sp.linalg.null_space(A)))
Vf

In [ ]:
Uf = np.hstack((Ured, sp.linalg.null_space(A.T)))
Uf

In [ ]:
Sf = np.zeros_like(A)  # full S matrix has same shape as A
Sf[0, 0], Sf[1, 1] = S1, S2  # two singular values along the diagonal
Sf

In [ ]:
# check with scipy's SVD
np.allclose(U, Uf), np.allclose(S, Sf), np.allclose(V, Vf)

In [ ]:
# check that matrices are unitary/orthonormal
np.allclose(sp.linalg.inv(V), V.T), \
    np.allclose(sp.linalg.inv(U), U.T), \
        V.T@V, V@V.T, U.T@U, U.T@U

# Projection Matrices For 4 Subspaces

In [ ]:
# projector to row space
P_RS = Vred @ Vred.T
P_RS

In [ ]:
# projector to column space
P_CS = Ured @ Ured.T
P_CS

In [ ]:
# projector to null space
P_NS = np.eye(N) - P_RS
P_NS

In [ ]:
# projector to left null space
P_LNS = np.eye(M) - P_CS
P_LNS

In [ ]:
# check the eigenvalue problems
print(sp.linalg.eig(P_RS)), print(sp.linalg.eig(P_NS))

In [ ]:
# check the eigenvalue problems
print(sp.linalg.eig(P_CS)), print(sp.linalg.eig(P_LNS))

In [ ]:
# check row/null projections
P_RS @ V, P_NS @ V

In [ ]:
# check column/leftnull projections
P_CS @ U, P_LNS @ U

# Pseudo-Inverse via SVD

In [ ]:
Sfinv = Sf.T
Sfinv[0, 0] = 1 / Sfinv[0, 0]  # invert the non-zero singular values
Sfinv[1, 1] = 1 / Sfinv[1, 1]
Sfinv

In [ ]:
Ainv = Vf @ Sfinv @ Uf.T
Ainv

In [ ]:
np.allclose(Ainv, sp.linalg.pinv(A))  # check with scipy's pinv

In [ ]:
np.allclose(Ainv @  A, P_RS)  # projector to row space

In [ ]:
np.allclose(A @ Ainv, P_CS)  # projector to column space
# A @ Ainv is often called hat-matrix in linear regression literature

# Task

The above matrix analysis should be done for the matrices A1, A2, A3 given below.

Probably it is a good idea to have no hard-coded parts as in the example above. Once knowing the rank of the matrix, it appears elegant to code everything with respect to this information, as it tells the dimensions and shapes of the subspaces, and thus of all involved matrices.

In [ ]:
A1 = np.array([
    [4, 0, 4],
    [2, 0, 2],
    [4, 2, 6],
    [4, 4, 8]
])
A1

In [ ]:
A2 = np.array([
    [9, 12, 3, 15],
    [0, 2, 0, 6],
    [9, 11, 3, 12]
])
A2

In [ ]:
A3 = np.array([
    [3, 0, 0],
    [0, 0, 2],
    [0, 1, 0]
])
A3

## Copyright

- the notebooks are provided as [Open Educational Resources](https://en.wikipedia.org/wiki/Open_educational_resources)
- the text is licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/)
- the code of the IPython examples is licensed under the [MIT license](https://opensource.org/licenses/MIT)
- feel free to use the notebooks for your own purposes
- please attribute the work as follows: *Frank Schultz, Data Driven Audio Signal Processing - A Tutorial Featuring Computational Examples, University of Rostock* ideally with relevant file(s), github URL https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise, commit number and/or version tag, year.